In [1]:
import os
import math
import random
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from tqdm import tqdm

In [2]:
class CFG:
    ROOT = "all_images"  

    IMG_SIZE = 224
    PATCH_SIZE = 16
    IN_CHANS = 3

    EMBED_DIM = 768        
    DEPTH = 12               
    NUM_HEADS = 6

    DECODER_DIM = 256
    DECODER_DEPTH = 4
    DECODER_HEADS = 4

    MASK_RATIO = 0.25

    BATCH_SIZE = 16
    EPOCHS = 30
    LR = 1e-4
    WEIGHT_DECAY = 0.05

    DINO_WEIGHT = 1.0
    MAE_WEIGHT = 1.0

    TEACHER_MOMENTUM = 0.996

    NUM_WORKERS = 4
    SAVE_DIR = "ssl_checkpoints"

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(CFG.SAVE_DIR, exist_ok=True)

In [ ]:
IMG_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]

def collect_images(root):
    root = Path(root)
    image_paths = []

    for p in root.rglob("*"):
        if p.suffix.lower() in IMG_EXTENSIONS:
            image_paths.append(str(p))

    return image_paths


image_paths = collect_images(CFG.ROOT)
print("Total ultrasound images found:", len(image_paths))

if len(image_paths) == 0:
    raise ValueError("No images found. Please check CFG.ROOT path.")

In [4]:
class UltrasoundDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths

        self.dino_transform_1 = T.Compose([
            T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.RandomRotation(30),
            T.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.7, 1.0)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])

        self.dino_transform_2 = T.Compose([
            T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.RandomRotation(30),
            T.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.6, 1.0)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])

        self.mae_transform = T.Compose([
            T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]

        image = Image.open(path).convert("RGB")

        view1 = self.dino_transform_1(image)
        view2 = self.dino_transform_2(image)
        mae_img = self.mae_transform(image)

        return {
            "view1": view1,
            "view2": view2,
            "mae_img": mae_img,
            "path": path
        }


dataset = UltrasoundDataset(image_paths)

loader = DataLoader(
    dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

print("SSL batches:", len(loader))

SSL batches: 23528


In [5]:
class PatchEmbedding(nn.Module):
    def __init__(self):
        super().__init__()

        self.grid_size = CFG.IMG_SIZE // CFG.PATCH_SIZE
        self.num_patches = self.grid_size * self.grid_size

        self.proj = nn.Conv2d(
            CFG.IN_CHANS,
            CFG.EMBED_DIM,
            kernel_size=CFG.PATCH_SIZE,
            stride=CFG.PATCH_SIZE
        )

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4.0):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=heads,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(dim)

        hidden_dim = int(dim * mlp_ratio)

        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim)
        )

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x, _ = self.attn(x, x, x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.mlp(x)
        x = x + shortcut

        return x

In [7]:

class ViTEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.patch_embed = PatchEmbedding()
        self.num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, CFG.EMBED_DIM))

        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.num_patches + 1, CFG.EMBED_DIM)
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                dim=CFG.EMBED_DIM,
                heads=CFG.NUM_HEADS
            )
            for _ in range(CFG.DEPTH)
        ])

        self.norm = nn.LayerNorm(CFG.EMBED_DIM)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x, return_tokens=False):
        B = x.size(0)

        patch_tokens = self.patch_embed(x)

        cls_token = self.cls_token.expand(B, -1, -1)

        x = torch.cat([cls_token, patch_tokens], dim=1)

        x = x + self.pos_embed

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        cls = x[:, 0]
        tokens = x[:, 1:]

        if return_tokens:
            return cls, tokens

        return cls

In [8]:

class MAEDecoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.num_patches = (CFG.IMG_SIZE // CFG.PATCH_SIZE) ** 2
        self.patch_dim = CFG.PATCH_SIZE * CFG.PATCH_SIZE * CFG.IN_CHANS

        self.decoder_embed = nn.Linear(CFG.EMBED_DIM, CFG.DECODER_DIM)

        self.mask_token = nn.Parameter(torch.zeros(1, 1, CFG.DECODER_DIM))

        self.decoder_pos_embed = nn.Parameter(
            torch.zeros(1, self.num_patches, CFG.DECODER_DIM)
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                dim=CFG.DECODER_DIM,
                heads=CFG.DECODER_HEADS
            )
            for _ in range(CFG.DECODER_DEPTH)
        ])

        self.norm = nn.LayerNorm(CFG.DECODER_DIM)

        self.pred = nn.Linear(CFG.DECODER_DIM, self.patch_dim)

        nn.init.trunc_normal_(self.mask_token, std=0.02)
        nn.init.trunc_normal_(self.decoder_pos_embed, std=0.02)

    def forward(self, visible_tokens, ids_restore):
        B, N_visible, D = visible_tokens.shape
        N = self.num_patches

        x = self.decoder_embed(visible_tokens)

        mask_tokens = self.mask_token.repeat(B, N - N_visible, 1)

        x_full = torch.cat([x, mask_tokens], dim=1)

        x_full = torch.gather(
            x_full,
            dim=1,
            index=ids_restore.unsqueeze(-1).repeat(1, 1, CFG.DECODER_DIM)
        )

        x_full = x_full + self.decoder_pos_embed

        for block in self.blocks:
            x_full = block(x_full)

        x_full = self.norm(x_full)

        pred = self.pred(x_full)

        return pred

In [9]:
class MAEModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = MAEDecoder()

    def patchify(self, imgs):
        p = CFG.PATCH_SIZE
        B, C, H, W = imgs.shape

        x = imgs.reshape(B, C, H // p, p, W // p, p)
        x = x.permute(0, 2, 4, 3, 5, 1)
        x = x.reshape(B, -1, p * p * C)

        return x

    def random_masking(self, x):
        B, N, D = x.shape

        len_keep = int(N * (1 - CFG.MASK_RATIO))

        noise = torch.rand(B, N, device=x.device)

        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)

        ids_keep = ids_shuffle[:, :len_keep]

        x_masked = torch.gather(
            x,
            dim=1,
            index=ids_keep.unsqueeze(-1).repeat(1, 1, D)
        )

        mask = torch.ones([B, N], device=x.device)
        mask[:, :len_keep] = 0

        mask = torch.gather(mask, dim=1, index=ids_restore)

        return x_masked, mask, ids_restore

    def forward(self, imgs):
        patch_tokens = self.encoder.patch_embed(imgs)

        patch_tokens = patch_tokens + self.encoder.pos_embed[:, 1:, :]

        visible_tokens, mask, ids_restore = self.random_masking(patch_tokens)

        for block in self.encoder.blocks:
            visible_tokens = block(visible_tokens)

        visible_tokens = self.encoder.norm(visible_tokens)

        pred = self.decoder(visible_tokens, ids_restore)

        target = self.patchify(imgs)

        loss = ((pred - target) ** 2).mean(dim=-1)

        loss = (loss * mask).sum() / mask.sum()

        return loss, pred, mask

In [10]:
class DINOHead(nn.Module):
    def __init__(self, in_dim, out_dim=4096):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_dim, 2048),
            nn.GELU(),
            nn.Linear(2048, 2048),
            nn.GELU(),
            nn.Linear(2048, out_dim)
        )

    def forward(self, x):
        return self.net(x)

In [11]:
class DINOLoss(nn.Module):
    def __init__(self, student_temp=0.1, teacher_temp=0.04):
        super().__init__()

        self.student_temp = student_temp
        self.teacher_temp = teacher_temp

    def forward(self, student_output, teacher_output):
        student_out = student_output / self.student_temp
        teacher_out = F.softmax(teacher_output / self.teacher_temp, dim=-1)

        loss = torch.sum(
            -teacher_out.detach() * F.log_softmax(student_out, dim=-1),
            dim=-1
        )

        return loss.mean()

In [12]:
class DINO_MAE_SSL(nn.Module):
    def __init__(self):
        super().__init__()

        self.student_encoder = ViTEncoder()
        self.teacher_encoder = ViTEncoder()

        self.student_head = DINOHead(CFG.EMBED_DIM)
        self.teacher_head = DINOHead(CFG.EMBED_DIM)

        self.mae = MAEModel(self.student_encoder)

        self.dino_loss = DINOLoss()

        self.teacher_encoder.load_state_dict(self.student_encoder.state_dict())
        self.teacher_head.load_state_dict(self.student_head.state_dict())

        for p in self.teacher_encoder.parameters():
            p.requires_grad = False

        for p in self.teacher_head.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update_teacher(self):
        m = CFG.TEACHER_MOMENTUM

        for ps, pt in zip(self.student_encoder.parameters(), self.teacher_encoder.parameters()):
            pt.data.mul_(m).add_((1 - m) * ps.data)

        for ps, pt in zip(self.student_head.parameters(), self.teacher_head.parameters()):
            pt.data.mul_(m).add_((1 - m) * ps.data)

    def forward(self, view1, view2, mae_img):
        student_feat = self.student_encoder(view1)
        student_out = self.student_head(student_feat)

        with torch.no_grad():
            teacher_feat = self.teacher_encoder(view2)
            teacher_out = self.teacher_head(teacher_feat)

        dino_loss = self.dino_loss(student_out, teacher_out)

        mae_loss, pred, mask = self.mae(mae_img)

        total_loss = CFG.DINO_WEIGHT * dino_loss + CFG.MAE_WEIGHT * mae_loss

        return total_loss, dino_loss, mae_loss

In [ ]:
model = DINO_MAE_SSL().to(CFG.DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.LR,
    weight_decay=CFG.WEIGHT_DECAY
)

scaler = torch.amp.GradScaler()

best_loss = float("inf")

for epoch in range(CFG.EPOCHS):
    model.train()

    total_loss = 0
    total_dino = 0
    total_mae = 0

    loop = tqdm(loader, desc=f"Epoch [{epoch+1}/{CFG.EPOCHS}]")

    for batch in loop:
        view1 = batch["view1"].to(CFG.DEVICE)
        view2 = batch["view2"].to(CFG.DEVICE)
        mae_img = batch["mae_img"].to(CFG.DEVICE)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            loss, dino_loss, mae_loss = model(view1, view2, mae_img)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        model.update_teacher()

        total_loss += loss.item()
        total_dino += dino_loss.item()
        total_mae += mae_loss.item()

        loop.set_postfix({
            "loss": loss.item(),
            "dino": dino_loss.item(),
            "mae": mae_loss.item()
        })

    avg_loss = total_loss / len(loader)
    avg_dino = total_dino / len(loader)
    avg_mae = total_mae / len(
        loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Total Loss: {avg_loss:.4f}")
    print(f"DINO Loss : {avg_dino:.4f}")
    print(f"MAE Loss  : {avg_mae:.4f}")

    checkpoint = {
    "epoch": epoch + 1,
    "student_encoder": model.student_encoder.state_dict(),
    "teacher_encoder": model.teacher_encoder.state_dict(),
    "student_head": model.student_head.state_dict(),
    "teacher_head": model.teacher_head.state_dict(),
    "mae_decoder": model.mae.decoder.state_dict(),
    "optimizer": optimizer.state_dict(),
    "loss": avg_loss,
    "config": {
        k: v
        for k, v in CFG.__dict__.items()
        if not k.startswith("__")
    }
    }
    
    torch.save(
        checkpoint,
        os.path.join(CFG.SAVE_DIR, "latest_dino_mae.pt")
    )

    print("Best SSL model saved.")

Epoch [1/30]:   0%|                                   | 0/23528 [00:00<?, ?it/s]/tmp/ipykernel_815183/2885574365.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch [1/30]: 100%|█| 23528/23528 [3:08:13<00:00,  2.08it/s, loss=0.26, dino=1.4



Epoch 1
Total Loss: 0.5274
DINO Loss : 0.0457
MAE Loss  : 0.4817
Best SSL model saved.


Epoch [2/30]: 100%|█| 23528/23528 [2:29:26<00:00,  2.62it/s, loss=0.0826, dino=1



Epoch 2
Total Loss: 0.1397
DINO Loss : 0.0044
MAE Loss  : 0.1353
Best SSL model saved.


Epoch [3/30]: 100%|█| 23528/23528 [2:25:07<00:00,  2.70it/s, loss=0.0645, dino=1


...

Epoch [5/30]: 100%|█| 23528/23528 [2:21:47<00:00,  2.77it/s, loss=0.556, dino=0.



Epoch 5
Total Loss: nan
DINO Loss : nan
MAE Loss  : 0.4067
Best SSL model saved.


Epoch [6/30]: 100%|█| 23528/23528 [2:22:33<00:00,  2.75it/s, loss=0.756, dino=0.



Epoch 6
Total Loss: 0.5649
DINO Loss : 0.0059
MAE Loss  : 0.5590
Best SSL model saved.


Epoch [7/30]:   5%| | 1120/23528 [07:05<2:16:20,  2.74it/s, loss=0.7, dino=0.008

In [ ]:
torch.save(
    {
        "encoder_state_dict": model.student_encoder.state_dict(),
        "img_size": CFG.IMG_SIZE,
        "patch_size": CFG.PATCH_SIZE,
        "embed_dim": CFG.EMBED_DIM
    },
    os.path.join(CFG.SAVE_DIR, "ultrasound_ssl_encoder.pt")
)

print("Final SSL encoder saved.")